In [1]:
import pandas as pd

# Getting the Scoring Sequence for each divison

In [2]:
series_df = pd.read_csv("../Data/mlb_division_series_results.csv")

# the CSV stores the series winner as a team code (or "TIE")
# the code below expects 1/-1/0
def get_series_result(row):
    if row["result"] == "TIE":
        return 0
    return 1 if row["result"] == row["team1"] else -1

series_df["result"] = series_df.apply(get_series_result, axis=1)

series_df = series_df[["season", "league", "division", "team1", "team2", "result", "wins1", "wins2"]]

# GET THE SERIES WINS FOR EACH TEAM

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    team1["result"] == 1
).astype(int)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    team2["result"] == -1
).astype(int)

# create a ranking
score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)


score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)

In [3]:
score_sequences.to_csv('../Analysis/mlb_scoring_sequence.csv', index=False)

# Getting upset frequencies per division

In [4]:
# lookup each team's score for a given season and division
score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# determine whether each series is an upset
def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)


# count upsets for each season and division
upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )["upset"]
    .sum()
)

In [5]:
upset_counts.to_csv('../Analysis/mlb_upset_counts.csv', index=False)